# Show1D

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/electronmicroscopy/quantem.widget/blob/main/docs/tutorials/show1d.ipynb)

`Show1D` is the iteration dashboard for traces, optimizer losses, image line profiles, and reconstruction snapshots. It is useful when a ptychography or denoising run produces scalar metrics every iteration and object/probe images at lower-rate checkpoints.

This tutorial uses the reconstruction snapshots for the synthetic Ducky example hosted in the public [`bobleesj/quantem-data`](https://huggingface.co/datasets/bobleesj/quantem-data) Hugging Face dataset. The default pins `widget/ducky`, and `show1d_ducky()` hides those storage details. See [Tutorial datasets](../api/datasets.md) for the immutable revision and canonical folder paths.


Ducky simulation tutorial: Georgios Varnavides, [*Trademark ducky dataset* (October 23, 2025)](https://github.com/electronmicroscopy/quantem-tutorials/blob/dbadc111734641226ab0454ed988feaf018e65c4/simulation_notebooks/ducky-dataset.ipynb). The viewer files contain subsequent reconstruction results for the synthetic example.


In [ ]:
# @title Install QuantEM { display-mode: "form" }
if get_ipython().__class__.__module__.startswith("google.colab"):
    !wget -q https://raw.githubusercontent.com/electronmicroscopy/quantem.widget/main/scripts/install_colab.py -O install_quantem.py
    %run install_quantem.py


In [ ]:
import quantem.widget as qw
qw.profile()

import json

import numpy as np

from quantem.widget import Show1D
from quantem.widget.datasets import show1d_ducky


## Compare loss traces with reconstruction snapshots

A joint reconstruction sweep often has many related loss curves: a frame-by-frame baseline plus several regularization strengths. The Ducky folder holds one JSON line per frame with every loss and the paths of that frame's snapshot images. The cell below plots the losses and attaches four representative checkpoints with four image candidates each, so this live documentation page remains compact.

Each snapshot group contains a reference image and the reconstruction candidates for the selected frame. The selected frame in the loss plot and the image grid stay linked. Use the star beside playback to bookmark an important iteration; the same action is available as `star_snapshot_group(...)` from Python.


In [ ]:
ducky_run = show1d_ducky(verbose=False)
events = [json.loads(line) for line in (ducky_run / "show1d_monitor.jsonl").read_text().splitlines()]
snapshot_names = ("reference", "lambda 0.3", "lambda 30", "lambda 300")

widget = Show1D(
    {name: [event["losses"][name] for event in events] for name in events[0]["losses"]},
    x=[event["iteration"] for event in events],
    title="Synthetic Ducky reconstructions (tutorial: Georgios Varnavides)",
    x_label="frame",
    y_label="loss",
    x_integer=True,
    snapshot_columns=4,
    snapshot_panel_width_px=720,
    snapshot_histogram_width=360,
    snapshot_histogram_height=52,
)
for event in events[::3]:
    widget.snapshot(
        event["iteration"],
        label=event["label"],
        **{name: np.load(ducky_run / event["snapshots"][name]) for name in snapshot_names},
    )
widget.star_snapshot_group(3)
widget


## Profile a single image

`Show1D.from_image` samples a line profile from a 2D image and keeps the image context beside the trace. Coordinates use the scientific `(row, col)` convention. Here we reuse the reference snapshot downloaded by `show1d_ducky()`.


In [ ]:
reference = np.load(ducky_run / "snapshots" / "reference.npy")

Show1D.from_image(
    reference,
    line=((70, 45), (170, 190)),
    profile_width=3,
    title="Ducky reference (tutorial: Georgios Varnavides)",
    image_cmap="viridis",
)
